In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'using Device: {device}')
print(f'CUDA Version: {torch.version.cuda}')

using Device: cuda
CUDA Version: 12.6


In [2]:
df = pd.read_csv('telco.csv')
df.columns = df.columns.str.strip()
df.head()

,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,...,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,...,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,...,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


In [3]:
df[['Offer', 'Internet Type', 'Churn Category', 'Churn Reason']] = df[['Offer', 'Internet Type', 'Churn Category', 'Churn Reason']].fillna('Unkown')

In [4]:
df['Churn Label'] = df['Churn Label'].replace({'No': 0, 'Yes':1})

C:\Users\Louis Bernal\AppData\Local\Temp\ipykernel_7892\2039206765.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Churn Label'] = df['Churn Label'].replace({'No': 0, 'Yes':1})


#### Data Preprocessing

In [5]:
label_encoder = LabelEncoder()
for col in df.select_dtypes(include='object').columns: 
    df[col] = label_encoder.fit_transform(df[col])

In [6]:
# Split data
x = df.drop(columns=['Churn Label']).values
y = df['Churn Label']

In [7]:
# normalize features
scaler = StandardScaler()
x = scaler.fit_transform(x)

In [8]:
# Convert to PyTorch tensors
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train, x_test = torch.tensor(x_train, dtype=torch.float32), torch.tensor(x_test, dtype=torch.float32)
y_train, y_test = torch.tensor(y_train.values, dtype=torch.float32), torch.tensor(y_test.values, dtype=torch.float32)

In [9]:
# define dataset class
class TelcoDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [10]:
# create dataholders

train_dataset = TelcoDataset(x_train, y_train)
test_dataset = TelcoDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [11]:
# define Neural Network
class TelcoModel(nn.Module):
    def __init__(self, input_dim):
        super(TelcoModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 1)
        self.relu =nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.dropout(self.relu(self.fc3(x)))
        x = self.sigmoid(self.fc4(x))
        return x

In [12]:
# initialize model, loss function and optmizer
model = TelcoModel(x_train.shape[1]).to(device)

# apply xavier intialization
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)

model.apply(init_weights)

# handle class imbalance
class_weights = torch.tensor([1.0, 3.0]).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
scheduler= StepLR(optimizer, step_size=10, gamma = 0.9)

In [ ]:
# Training loop with early stopping
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()  
        running_loss += loss.item()  
    
    # Adjust learning rate
    scheduler.step()

    if epoch % 10 == 0:
        print(f'Evaluating Epoch {epoch}')
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs).squeeze()
                predicted = (outputs > 0.5).float()
                correct += (predicted == labels).sum().item()  # Fixed: `.item()`
                total += labels.size(0)
        
        accuracy = correct / total
        print(f'Epoch {epoch}: Loss {running_loss/len(train_loader):.4f}, Accuracy= {accuracy:.4f}')


Evaluating Epoch 0
Epoch 0: Loss 0.4454, Accuracy= 0.9957
Evaluating Epoch 10
Epoch 10: Loss 0.0143, Accuracy= 1.0000
